# Reversible LLM training

**Task (study guide §10):** train a ~20M-parameter LLM for 50M tokens, three runs:
1. **Baseline** — standard residual transformer at a fixed batch (32 × 512).
2. **Reversible, same batch** — and report which integrator variant worked (midpoint/leapfrog vs reversible Euler).
3. **Reversible, max batch** — push the batch as large as the freed memory allows.

Report: final loss, speed (tokens/s), peak memory, findings.

| Part | What |
|---|---|
| 0 | Setup (GPU, precision, config) |
| 1 | Data: TinyStories → 8K BPE → `data/train.bin`, `data/val.bin` |
| 2 | Code: `reversible.py`, `model.py`, `train.py`, `find_max_batch.py`, `make_report.py`, written with `%%writefile`. These are the exact files that were tested |
| 3 | Tests: reversible gradients == autograd, reconstruction, memory flat in depth |
| 4 | Integrator screening (short runs) |
| 5 | The three assignment runs (+ optional extras) |
| 6 | Report: tables + plots |
| 7 | Results obtained on RTX 5070 Ti + findings |

Run top to bottom on **Google Colab (Runtime → Change runtime type → GPU)** or on a local CUDA machine.
On a T4 the full notebook takes roughly 1.5–2.5 h; turn off the optional parts in the config cell to save time.

## 0. Setup

In [ ]:
import os, sys, json, glob, torch
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    !pip -q install tokenizers datasets
    os.makedirs("/content/era13", exist_ok=True)
    %cd /content/era13
os.makedirs("tests", exist_ok=True)
assert torch.cuda.is_available(), "Needs a CUDA GPU (Colab: Runtime > Change runtime type > GPU)"
DT = "bf16" if torch.cuda.get_device_capability()[0] >= 8 else "fp16"      # native bf16 needs Ampere+; T4 would emulate it (very slow) -> fp16 + GradScaler
print(torch.__version__, "|", torch.cuda.get_device_name(), "| autocast dtype:", DT)

# ---- experiment config ----
B0 = 32                 # baseline batch (x 512 tokens)
TOKENS = 50e6           # token budget per run
RUN_SCREENING = True    # 5M-token integrator screening (7 short runs)
RUN_EXTRAS = True       # Run 1b (baseline at max batch) and Run 3b (fp32 stream at max batch)
HEADROOM = 0.90 if IN_COLAB else 0.80   # fraction of free VRAM for the max-batch probe (Windows spills to host RAM near full)

## 1. Data
TinyStories, tokenised with an **8,192-vocab byte-level BPE**. A small vocab keeps the embedding/logits from dominating
parameters and peak memory, so the activation savings being measured stay visible. If `data/raw/*.parquet` exists
(curl-downloaded), it is used; otherwise the script streams from the Hugging Face Hub.

### `prepare_data.py`

In [ ]:
%%writefile prepare_data.py
"""Download TinyStories, train an 8K byte-level BPE, and write uint16 token files.

Outputs (in data/):
  tokenizer.json   - the trained BPE
  train.bin        - >= --train_tokens tokens (uint16)
  val.bin          - --val_tokens tokens from the validation split (uint16)
"""
import argparse
import os

import numpy as np
from datasets import load_dataset
from tokenizers import Tokenizer, decoders, models, pre_tokenizers, trainers

EOS = "<|endoftext|>"
RAW = {"train": "data/raw/train0.parquet", "validation": "data/raw/val.parquet"}


def stories(split):
    """Stream TinyStories; prefer local parquet (curl-downloaded) over the Hub."""
    if os.path.exists(RAW[split]):
        return load_dataset("parquet", data_files=RAW[split], split="train", streaming=True)
    return load_dataset("roneneldan/TinyStories", split=split, streaming=True)


def train_tokenizer(texts, vocab_size):
    tok = Tokenizer(models.BPE())
    tok.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)
    tok.decoder = decoders.ByteLevel()
    trainer = trainers.BpeTrainer(
        vocab_size=vocab_size,
        special_tokens=[EOS],
        initial_alphabet=pre_tokenizers.ByteLevel.alphabet(),
    )
    tok.train_from_iterator(texts, trainer=trainer)
    return tok


def write_tokens(tok, stream, path, target, batch=2000):
    eos_id = tok.token_to_id(EOS)
    buf = np.empty(target + 1_000_000, dtype=np.uint16)
    n = 0
    texts = []

    def flush():
        nonlocal n
        for enc in tok.encode_batch(texts):
            ids = enc.ids + [eos_id]
            k = min(len(ids), len(buf) - n)
            buf[n:n + k] = ids[:k]
            n += k
        texts.clear()

    for ex in stream:
        texts.append(ex["text"])
        if len(texts) == batch:
            flush()
            print(f"\r{path}: {n / 1e6:.1f}M tokens", end="", flush=True)
            if n >= target:
                break
    if texts and n < target:
        flush()
    print()
    buf[:n].tofile(path)
    return n


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--out", default="data")
    ap.add_argument("--vocab", type=int, default=8192)
    ap.add_argument("--tok_docs", type=int, default=200_000)
    ap.add_argument("--train_tokens", type=int, default=55_000_000)
    ap.add_argument("--val_tokens", type=int, default=1_000_000)
    args = ap.parse_args()
    os.makedirs(args.out, exist_ok=True)

    tok_path = os.path.join(args.out, "tokenizer.json")
    if os.path.exists(tok_path):
        tok = Tokenizer.from_file(tok_path)
    else:
        ds = stories("train")
        texts = [ex["text"] for _, ex in zip(range(args.tok_docs), ds)]
        tok = train_tokenizer(texts, args.vocab)
        tok.save(tok_path)
    print("vocab size:", tok.get_vocab_size())

    train = stories("train")
    n = write_tokens(tok, train, os.path.join(args.out, "train.bin"), args.train_tokens)
    print(f"train tokens: {n:,}")
    val = stories("validation")
    n = write_tokens(tok, val, os.path.join(args.out, "val.bin"), args.val_tokens)
    print(f"val tokens: {n:,}")


if __name__ == "__main__":
    main()

In [ ]:
if not os.path.exists("data/train.bin"):
    !python prepare_data.py
!ls -la data

## 2. Code
### `reversible.py`: memory-free backward
The forward runs under `no_grad` and keeps only the final state(s). The backward walks the layers in reverse,
**reconstructs** each layer's input from its output, recomputes that single layer with grad enabled, and back-propagates
through it. Activation memory is therefore one layer, independent of depth.

* **Midpoint / leapfrog:** `x_{l+1} = x_{l-1} + 2h f_l(x_l)`; inverse `x_{l-1} = x_{l+1} - 2h f_l(x_l)`
* **Reversible (symplectic) Euler / RevNet coupling:** `y' = y + h·Attn(z)`, `z' = z + h·MLP(y')`; inverse `z = z' - h·MLP(y')`, `y = y' - h·Attn(z)`

In [ ]:
%%writefile reversible.py
"""Memory-free backward passes for reversible trunks.

The forward runs under no_grad and keeps only the final state(s). The backward
walks the layers in reverse, *reconstructing* each layer's input from its output,
recomputing that single layer with grad enabled, and back-propagating through it.
Peak activation memory is therefore one layer, independent of depth.
"""
import torch

_fwd = torch.amp.custom_fwd(device_type="cuda")
_bwd = torch.amp.custom_bwd(device_type="cuda")


def _grads(out, inputs, grad_out):
    gs = torch.autograd.grad(out, inputs, grad_out, allow_unused=True)
    return [torch.zeros_like(i) if g is None else g for g, i in zip(gs, inputs)]


class RevMidpointFn(torch.autograd.Function):
    """Leapfrog: x_{l+1} = x_{l-1} + 2h f_l(x_l);  inverse x_{l-1} = x_{l+1} - 2h f_l(x_l)."""

    @staticmethod
    @_fwd
    def forward(ctx, x0, x1, h, blocks, *params):
        with torch.no_grad():
            prev, cur = x0, x1
            for b in blocks:
                prev, cur = cur, prev + 2 * h * b.delta(cur)
        ctx.save_for_backward(prev, cur)
        ctx.h, ctx.blocks = h, blocks
        return cur

    @staticmethod
    @_bwd
    def backward(ctx, g_out):
        a, b = ctx.saved_tensors          # (x_l, x_{l+1}), starting at l = L-1
        ga, gb = torch.zeros_like(g_out), g_out
        h, blocks = ctx.h, ctx.blocks
        param_grads = []
        for blk in reversed(blocks):
            params = list(blk.parameters())
            with torch.enable_grad():
                a_ = a.detach().requires_grad_()
                y = blk.delta(a_)
            gs = _grads(y, [a_] + params, 2 * h * gb)
            x_prev = b - 2 * h * y.detach()          # reconstruct x_{l-1}
            # pair becomes (x_{l-1}, x_l); x_{l-1} feeds x_{l+1} with identity
            a, b = x_prev, a
            ga, gb = gb, ga + gs[0]
            param_grads.append(gs[1:])
        flat = [g for pg in reversed(param_grads) for g in pg]
        return (ga, gb, None, None, *flat)


class RevEulerFn(torch.autograd.Function):
    """Two-stream symplectic Euler (RevNet coupling):
    y' = y + h F(z);  z' = z + h G(y')   with exact inverse
    z = z' - h G(y');  y = y' - h F(z).
    """

    @staticmethod
    @_fwd
    def forward(ctx, y, z, h, blocks, *params):
        with torch.no_grad():
            for b in blocks:
                y = y + h * b.attn_delta(z)
                z = z + h * b.mlp_delta(y)
        ctx.save_for_backward(y, z)
        ctx.h, ctx.blocks = h, blocks
        return y, z

    @staticmethod
    @_bwd
    def backward(ctx, gy, gz):
        y, z = ctx.saved_tensors
        h, blocks = ctx.h, ctx.blocks
        gy = torch.zeros_like(y) if gy is None else gy
        gz = torch.zeros_like(z) if gz is None else gz
        param_grads = []
        for blk in reversed(blocks):
            params = list(blk.parameters())
            # undo z' = z + h G(y')
            with torch.enable_grad():
                y_ = y.detach().requires_grad_()
                m = blk.mlp_delta(y_)
            gm = _grads(m, [y_] + params, h * gz)
            z = z - h * m.detach()
            gy = gy + gm[0]
            # undo y' = y + h F(z)
            with torch.enable_grad():
                z_ = z.detach().requires_grad_()
                a = blk.attn_delta(z_)
            ga = _grads(a, [z_] + params, h * gy)
            y = y - h * a.detach()
            gz = gz + ga[0]
            param_grads.append([p1 + p2 for p1, p2 in zip(gm[1:], ga[1:])])
        flat = [g for pg in reversed(param_grads) for g in pg]
        return (gy, gz, None, None, *flat)


@torch.no_grad()
def reconstruction_error(model, idx):
    """Run the reversible trunk forward then invert it; return ||x0_hat - x0|| / ||x0||."""
    cfg, blocks = model.cfg, model.blocks
    T = idx.shape[1]
    x0 = model.tok_emb(idx) + model.pos_emb(torch.arange(T, device=idx.device))
    x0 = x0.to(torch.float64 if cfg.stream_dtype == "fp64" else torch.float32)
    h = cfg.h
    if cfg.trunk.startswith("midpoint"):
        prev, cur = x0, x0 + h * blocks[0].delta(x0)
        x1 = cur
        for b in blocks[1:]:
            prev, cur = cur, prev + 2 * h * b.delta(cur)
        for b in reversed(blocks[1:]):
            prev, cur = cur - 2 * h * b.delta(prev), prev
        return max(((prev - x0).norm() / x0.norm()).item(), ((cur - x1).norm() / x1.norm()).item())
    y = z = x0
    for b in blocks:
        y = y + h * b.attn_delta(z)
        z = z + h * b.mlp_delta(y)
    for b in reversed(blocks):
        z = z - h * b.mlp_delta(y)
        y = y - h * b.attn_delta(z)
    return max(((y - x0).norm() / x0.norm()).item(), ((z - x0).norm() / x0.norm()).item())

### `model.py`: GPT with interchangeable trunks
d=384, 10 layers, 6 heads, context 512, tied embeddings, no dropout (the backward must recompute the same function) → **21.05M params**.
`residual` is the baseline; `midpoint` / `reveuler` are the same architecture with stored activations; `*_rev` use the memory-free backward.
`stream_dtype="fp64"` makes reconstruction exact. The loss is a checkpointed, chunked cross-entropy, the same for all runs.

In [ ]:
%%writefile model.py
"""Small GPT with interchangeable trunks.

trunk options
  residual       standard pre-LN transformer (baseline)
  midpoint       reversible-midpoint (leapfrog) architecture, ordinary autograd (stores activations)
  midpoint_rev   same network, memory-free backward (reconstructs states in reverse)
  reveuler       two-stream reversible (symplectic) Euler / RevNet coupling, ordinary autograd
  reveuler_rev   same network, memory-free backward
"""
import math
from dataclasses import dataclass

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.checkpoint import checkpoint

from reversible import RevEulerFn, RevMidpointFn

TRUNKS = ("residual", "midpoint", "midpoint_rev", "reveuler", "reveuler_rev")


@dataclass
class GPTConfig:
    vocab_size: int = 8192
    seq_len: int = 512
    n_layer: int = 10
    n_head: int = 6
    d_model: int = 384
    trunk: str = "residual"
    h: float = 0.5  # integrator step size (midpoint uses 2h per layer)
    loss_chunk: int = 8192  # tokens per checkpointed loss chunk (0 = plain CE)
    stream_dtype: str = "fp32"  # residual-stream precision; fp64 makes reverse reconstruction ~exact


class Attention(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.n_head = cfg.n_head
        self.qkv = nn.Linear(cfg.d_model, 3 * cfg.d_model, bias=False)
        self.proj = nn.Linear(cfg.d_model, cfg.d_model, bias=False)

    def forward(self, x):
        B, T, C = x.shape
        q, k, v = self.qkv(x).view(B, T, 3, self.n_head, C // self.n_head).permute(2, 0, 3, 1, 4)
        y = F.scaled_dot_product_attention(q, k, v, is_causal=True)
        return self.proj(y.transpose(1, 2).reshape(B, T, C))


class MLP(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.fc = nn.Linear(cfg.d_model, 4 * cfg.d_model, bias=False)
        self.proj = nn.Linear(4 * cfg.d_model, cfg.d_model, bias=False)

    def forward(self, x):
        return self.proj(F.gelu(self.fc(x)))


def _compute_in(x, like):
    """Blocks compute in the parameter dtype even when the residual stream is fp64."""
    return x.to(like.dtype)


class Block(nn.Module):
    """A transformer block exposed as residual *deltas* so integrators can compose them."""

    def __init__(self, cfg):
        super().__init__()
        self.ln1 = nn.LayerNorm(cfg.d_model)
        self.attn = Attention(cfg)
        self.ln2 = nn.LayerNorm(cfg.d_model)
        self.mlp = MLP(cfg)

    def attn_delta(self, x):
        return self.attn(self.ln1(_compute_in(x, self.ln1.weight))).to(x.dtype)

    def mlp_delta(self, x):
        return self.mlp(self.ln2(_compute_in(x, self.ln2.weight))).to(x.dtype)

    def delta(self, x):
        """f(x) = Block(x) - x for a standard pre-LN block."""
        a = self.attn_delta(x)
        return a + self.mlp_delta(x + a)


class GPT(nn.Module):
    def __init__(self, cfg: GPTConfig):
        super().__init__()
        assert cfg.trunk in TRUNKS, cfg.trunk
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.d_model)
        self.pos_emb = nn.Embedding(cfg.seq_len, cfg.d_model)
        self.blocks = nn.ModuleList(Block(cfg) for _ in range(cfg.n_layer))
        self.ln_f = nn.LayerNorm(cfg.d_model)
        self.head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)
        self.head.weight = self.tok_emb.weight  # tied
        self.apply(self._init)
        for name, p in self.named_parameters():
            if name.endswith("proj.weight"):
                nn.init.normal_(p, std=0.02 / math.sqrt(2 * cfg.n_layer))

    @staticmethod
    def _init(m):
        if isinstance(m, (nn.Linear, nn.Embedding)):
            nn.init.normal_(m.weight, std=0.02)

    def num_params(self):
        return sum(p.numel() for p in self.parameters())

    def block_params(self):
        return [p for b in self.blocks for p in b.parameters()]

    # ---- trunks -------------------------------------------------------
    def trunk(self, x):
        t, h, blocks = self.cfg.trunk, self.cfg.h, self.blocks
        if t == "residual":
            for b in blocks:
                x = x + b.delta(x)
            return x
        if t.startswith("midpoint"):
            # Euler starter step, then leapfrog: x_{l+1} = x_{l-1} + 2h f_l(x_l)
            x1 = x + h * blocks[0].delta(x)
            if t == "midpoint_rev":
                params = [p for b in blocks[1:] for p in b.parameters()]
                return RevMidpointFn.apply(x, x1, h, blocks[1:], *params)
            prev, cur = x, x1
            for b in blocks[1:]:
                prev, cur = cur, prev + 2 * h * b.delta(cur)
            return cur
        # reveuler: y' = y + h F(z);  z' = z + h G(y')
        if t == "reveuler_rev":
            y, z = RevEulerFn.apply(x, x, h, blocks, *self.block_params())
        else:
            y = z = x
            for b in blocks:
                y = y + h * b.attn_delta(z)
                z = z + h * b.mlp_delta(y)
        return 0.5 * (y + z)

    def forward(self, idx, targets=None):
        T = idx.shape[1]
        x = self.tok_emb(idx) + self.pos_emb(torch.arange(T, device=idx.device))
        sd = torch.float64 if self.cfg.stream_dtype == "fp64" else torch.float32
        x = x.to(torch.promote_types(x.dtype, sd))
        x = _compute_in(self.trunk(x), self.ln_f.weight)
        if targets is None:
            return self.head(self.ln_f(x))
        x, targets = x.reshape(-1, x.size(-1)), targets.reshape(-1)
        c = self.cfg.loss_chunk
        if c <= 0:
            return self._ce_sum(x, targets) / targets.numel()
        # checkpointed chunks: only one chunk's logits ever exist (same for every trunk)
        total = sum(checkpoint(self._ce_sum, x[i:i + c], targets[i:i + c], use_reentrant=False)
                    for i in range(0, targets.numel(), c))
        return total / targets.numel()

    def _ce_sum(self, x, targets):
        return F.cross_entropy(self.head(self.ln_f(x)).float(), targets, reduction="sum")

### `train.py`: token-budgeted training; records val loss, tokens/s, peak memory

In [ ]:
%%writefile train.py
"""Train the 20M GPT on a fixed token budget and record loss / tokens/s / peak memory.

Example:
  python train.py --name run1_baseline --trunk residual --batch 32
  python train.py --name run2_rev_same --trunk midpoint_rev --batch 32
"""
import argparse
import csv
import json
import math
import os
import time

import numpy as np
import torch

from model import GPT, TRUNKS, GPTConfig
from reversible import reconstruction_error


def get_args(argv=None):
    ap = argparse.ArgumentParser()
    ap.add_argument("--name", required=True)
    ap.add_argument("--trunk", choices=TRUNKS, default="residual")
    ap.add_argument("--h", type=float, default=None, help="step size (default 0.5 midpoint, 1.0 reveuler)")
    ap.add_argument("--stream", choices=["fp32", "fp64"], default="fp32", help="residual-stream dtype")
    ap.add_argument("--batch", type=int, default=32)
    ap.add_argument("--seq", type=int, default=512)
    ap.add_argument("--tokens", type=float, default=50e6)
    ap.add_argument("--lr", type=float, default=1e-3)
    ap.add_argument("--scale_lr", action="store_true", help="lr *= sqrt(batch/32), capped at 2x")
    ap.add_argument("--warmup", type=float, default=0.02)
    ap.add_argument("--wd", type=float, default=0.1)
    ap.add_argument("--dtype", choices=["bf16", "fp16"], default="bf16")
    ap.add_argument("--val_tokens", type=float, default=500_000)
    ap.add_argument("--log_every", type=int, default=25)
    ap.add_argument("--seed", type=int, default=1337)
    ap.add_argument("--data", default="data")
    ap.add_argument("--out", default="results")
    return ap.parse_args(argv)


def default_h(trunk):
    return 1.0 if trunk.startswith("reveuler") else 0.5


class Loader:
    def __init__(self, path, batch, seq, seed):
        self.data = np.memmap(path, dtype=np.uint16, mode="r")
        self.batch, self.seq = batch, seq
        self.rng = np.random.default_rng(seed)

    def next(self):
        ix = self.rng.integers(0, len(self.data) - self.seq - 1, self.batch)
        x = np.stack([self.data[i:i + self.seq + 1] for i in ix]).astype(np.int64)
        x = torch.from_numpy(x).pin_memory().cuda(non_blocking=True)
        return x[:, :-1], x[:, 1:]


@torch.no_grad()
def evaluate(model, path, seq, n_tokens, batch, amp):
    data = np.memmap(path, dtype=np.uint16, mode="r")
    n_win = min(int(n_tokens) // seq, (len(data) - 1) // seq)
    model.eval()
    tot = 0.0
    for s in range(0, n_win, batch):
        ix = range(s, min(s + batch, n_win))
        x = np.stack([data[i * seq:i * seq + seq + 1] for i in ix]).astype(np.int64)
        x = torch.from_numpy(x).cuda()
        with amp:
            tot += model(x[:, :-1], x[:, 1:]).item() * len(ix)
    model.train()
    return tot / n_win


def lr_at(step, total, peak, warmup):
    w = max(1, int(warmup * total))
    if step < w:
        return peak * (step + 1) / w
    p = (step - w) / max(1, total - w)
    return peak * (0.1 + 0.9 * 0.5 * (1 + math.cos(math.pi * p)))


def train(args):
    torch.manual_seed(args.seed)
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    h = args.h if args.h is not None else default_h(args.trunk)
    cfg = GPTConfig(seq_len=args.seq, trunk=args.trunk, h=h, stream_dtype=args.stream)
    model = GPT(cfg).cuda()
    n_params = model.num_params()

    lr = args.lr * (min(2.0, math.sqrt(args.batch / 32)) if args.scale_lr else 1.0)
    decay = [p for p in model.parameters() if p.dim() >= 2]
    no_decay = [p for p in model.parameters() if p.dim() < 2]
    opt = torch.optim.AdamW([{"params": decay, "weight_decay": args.wd},
                             {"params": no_decay, "weight_decay": 0.0}],
                            lr=lr, betas=(0.9, 0.95), fused=True)
    dt = torch.bfloat16 if args.dtype == "bf16" else torch.float16
    amp = torch.autocast("cuda", dtype=dt)
    scaler = torch.amp.GradScaler("cuda", enabled=args.dtype == "fp16")

    tok_per_step = args.batch * args.seq
    steps = math.ceil(args.tokens / tok_per_step)
    loader = Loader(os.path.join(args.data, "train.bin"), args.batch, args.seq, args.seed)
    os.makedirs(args.out, exist_ok=True)
    log_f = open(os.path.join(args.out, f"{args.name}.csv"), "w", newline="")
    log = csv.writer(log_f)
    log.writerow(["step", "tokens", "loss", "lr", "tok_per_s", "elapsed_s"])
    print(f"[{args.name}] trunk={args.trunk} h={h} stream={args.stream} params={n_params / 1e6:.2f}M "
          f"batch={args.batch}x{args.seq} steps={steps} lr={lr:.2e}")

    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    ema, diverged, warm_t, warm_step = None, False, None, 20
    torch.cuda.synchronize()
    t0 = t_log = time.perf_counter()
    step_log = 0
    for step in range(steps):
        for g in opt.param_groups:
            g["lr"] = lr_at(step, steps, lr, args.warmup)
        x, y = loader.next()
        with amp:
            loss = model(x, y)
        scaler.scale(loss).backward()
        scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(opt)
        scaler.update()
        opt.zero_grad(set_to_none=True)

        if step == warm_step - 1:
            torch.cuda.synchronize()
            warm_t = time.perf_counter()
        if (step + 1) % args.log_every == 0 or step == steps - 1:
            li = loss.item()
            if not math.isfinite(li):
                diverged = True
                print(f"  step {step + 1}: loss is {li}, stopping")
                break
            ema = li if ema is None else 0.9 * ema + 0.1 * li
            now = time.perf_counter()
            tps = (step + 1 - step_log) * tok_per_step / (now - t_log)  # last window may be < log_every steps
            t_log, step_log = now, step + 1
            log.writerow([step + 1, (step + 1) * tok_per_step, f"{li:.4f}", f"{opt.param_groups[0]['lr']:.3e}",
                          f"{tps:.0f}", f"{now - t0:.1f}"])
            if (step + 1) % (args.log_every * 8) == 0 or step == steps - 1:
                log_f.flush()
                print(f"  step {step + 1}/{steps} loss {li:.4f} ema {ema:.4f} tok/s {tps:,.0f} "
                      f"mem {torch.cuda.max_memory_allocated() / 2**30:.2f}GiB")
    torch.cuda.synchronize()
    t1 = time.perf_counter()
    log_f.close()
    steps_done = step + 1
    peak_alloc = torch.cuda.max_memory_allocated() / 2**30
    peak_res = torch.cuda.max_memory_reserved() / 2**30

    val = float("nan") if diverged else evaluate(model, os.path.join(args.data, "val.bin"), args.seq,
                                                 args.val_tokens, 32, amp)
    rec = None
    if args.trunk.endswith("_rev"):
        with amp:
            rec = reconstruction_error(model, x[:4])
    res = dict(
        name=args.name, trunk=args.trunk, h=h, stream=args.stream, batch=args.batch, seq=args.seq, params=n_params,
        lr=lr, steps=steps_done, tokens=steps_done * tok_per_step,
        final_train_loss=ema, val_loss=val, diverged=diverged,
        tokens_per_s=steps_done * tok_per_step / (t1 - t0),
        tokens_per_s_steady=((steps_done - warm_step) * tok_per_step / (t1 - warm_t)) if warm_t else None,
        wall_s=t1 - t0, peak_mem_gib=peak_alloc, peak_reserved_gib=peak_res, recon_error=rec,
        gpu=torch.cuda.get_device_name(), dtype=args.dtype,
    )
    with open(os.path.join(args.out, f"{args.name}.json"), "w") as f:
        json.dump(res, f, indent=2)
    print(json.dumps(res, indent=2))
    return res


if __name__ == "__main__":
    train(get_args())

### `find_max_batch.py`: largest batch that trains (doubling + binary search with real optimizer steps)

In [ ]:
%%writefile find_max_batch.py
"""Find the largest batch (multiple of --granularity) that trains without OOM.

Each probe builds the model + fused AdamW and runs a few full training steps,
so the peak includes weights, grads, optimizer state and activations.
Example:  python find_max_batch.py --trunk midpoint_rev --stream fp64
"""
import argparse
import gc
import json
import os

import torch

from model import GPT, TRUNKS, GPTConfig
from train import default_h


def probe(trunk, batch, seq, h, stream, dtype=torch.bfloat16, steps=3):
    torch.manual_seed(0)
    model = opt = x = loss = None
    try:
        model = GPT(GPTConfig(seq_len=seq, trunk=trunk, h=h, stream_dtype=stream)).cuda()
        opt = torch.optim.AdamW(model.parameters(), lr=1e-4, fused=True)
        torch.cuda.reset_peak_memory_stats()
        for _ in range(steps):
            x = torch.randint(0, model.cfg.vocab_size, (batch, seq + 1), device="cuda")
            with torch.autocast("cuda", dtype=dtype):
                loss = model(x[:, :-1], x[:, 1:])
            loss.backward()
            opt.step()
            opt.zero_grad(set_to_none=True)
        torch.cuda.synchronize()
        return torch.cuda.max_memory_reserved()
    except torch.cuda.OutOfMemoryError:
        return None
    finally:
        del model, opt, x, loss
        gc.collect()
        torch.cuda.empty_cache()


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--trunk", choices=TRUNKS, required=True)
    ap.add_argument("--h", type=float, default=None)
    ap.add_argument("--stream", choices=["fp32", "fp64"], default="fp32")
    ap.add_argument("--seq", type=int, default=512)
    ap.add_argument("--start", type=int, default=32)
    ap.add_argument("--granularity", type=int, default=8)
    ap.add_argument("--headroom", type=float, default=0.80,
                    help="fraction of free memory allowed; near-full VRAM on Windows spills to host RAM and runs slow")
    ap.add_argument("--dtype", choices=["bf16", "fp16"], default="bf16", help="fp16 on GPUs without bf16 (T4)")
    ap.add_argument("--out", default="results")
    args = ap.parse_args()
    dt = torch.bfloat16 if args.dtype == "bf16" else torch.float16
    h = args.h if args.h is not None else default_h(args.trunk)

    free, _ = torch.cuda.mem_get_info()
    budget = args.headroom * free
    fits = lambda b: (lambda r: r is not None and r <= budget)(probe(args.trunk, b, args.seq, h, args.stream, dt))

    lo, hi = 0, args.start
    while fits(hi):
        print(f"  batch {hi}: fits")
        lo, hi = hi, hi * 2
    print(f"  batch {hi}: does not fit")
    g = args.granularity
    while hi - lo > g:
        mid = (lo + hi) // 2 // g * g
        if mid <= lo:
            break
        ok = fits(mid)
        print(f"  batch {mid}: {'fits' if ok else 'does not fit'}")
        lo, hi = (mid, hi) if ok else (lo, mid)
    print(f"max batch for {args.trunk} (stream={args.stream}): {lo}  (budget {budget / 2**30:.1f} GiB)")
    os.makedirs(args.out, exist_ok=True)
    with open(os.path.join(args.out, f"maxbatch_{args.trunk}_{args.stream}.json"), "w") as f:
        json.dump(dict(trunk=args.trunk, stream=args.stream, h=h, seq=args.seq, max_batch=lo,
                       budget_gib=budget / 2**30), f, indent=2)


if __name__ == "__main__":
    main()

### `make_report.py`: tables + plots

In [ ]:
%%writefile make_report.py
"""Collect results/*.json into REPORT_results.md + plots (results/plots/*.png)."""
import csv
import glob
import json
import os

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt  # noqa: E402

MAIN = ["run1_baseline", "run2_rev_same", "run3_rev_max", "run1b_baseline_max", "run3b_rev_max_fp32stream",
        "colab_run3_rev_max_fp64"]
LABEL = {"run1_baseline": "Run 1 baseline", "run2_rev_same": "Run 2 reversible, same batch",
         "run3_rev_max": "Run 3 reversible, max batch", "run1b_baseline_max": "Run 1b baseline, max batch",
         "colab_run3_rev_max_fp64": "Colab Run 3 reference (fp64 stream)",
         "run3b_rev_max_fp32stream": "Run 3b reversible, max batch, fp32 stream"}


def load(pattern):
    return {os.path.basename(p)[:-5]: json.load(open(p)) for p in sorted(glob.glob(pattern))}


def fmt(v, f="{:.3f}"):
    return "—" if v is None else f.format(v)


def row(name, r):
    return (f"| {name} | {r['trunk']} | {r['h'] if 'rev' in r['trunk'] or 'mid' in r['trunk'] else '—'} "
            f"| {r.get('stream', 'fp32')} | {r['batch']} | {r['steps']} | {fmt(r['final_train_loss'])} "
            f"| {fmt(r['val_loss'])} | {r['tokens_per_s']:,.0f} | {fmt(r['tokens_per_s_steady'], '{:,.0f}')} "
            f"| {r['peak_mem_gib']:.2f} | {fmt(r['recon_error'], '{:.1e}')} | {r.get('gpu', '').replace('NVIDIA ', '')} |")


HDR = ("| run | trunk | h | stream | batch | steps | final train loss | val loss | tokens/s | tokens/s (steady) "
       "| peak mem (GiB) | recon err |\n|---|---|---|---|---|---|---|---|---|---|---|---|")


def curves(runs, folder, path, title):
    plt.figure(figsize=(7, 4))
    for name in runs:
        f = os.path.join(folder, f"{name}.csv")
        if not os.path.exists(f):
            continue
        rows = list(csv.DictReader(open(f)))
        plt.plot([int(r["tokens"]) / 1e6 for r in rows], [float(r["loss"]) for r in rows],
                 label=LABEL.get(name, name), lw=1)
    plt.xlabel("tokens (M)")
    plt.ylabel("train loss")
    plt.ylim(1.5, 6.0)
    plt.title(title)
    plt.legend()
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(path, dpi=120)
    plt.close()


def bars(res, path):
    gpu = res.get("run1_baseline", {}).get("gpu")
    names = [n for n in MAIN if n in res and res[n].get("gpu") == gpu]  # tokens/s only comparable on one GPU
    fig, ax = plt.subplots(1, 2, figsize=(10, 3.8))
    lbl = [f"{LABEL[n].split(' ', 2)[1]}\n{res[n]['trunk']}\nB={res[n]['batch']}" for n in names]
    ax[0].bar(lbl, [res[n]["tokens_per_s_steady"] / 1e3 for n in names], color="#4C78A8")
    ax[0].set_title("throughput (k tokens/s, steady)")
    ax[1].bar(lbl, [res[n]["peak_mem_gib"] for n in names], color="#F58518")
    ax[1].set_title("peak memory allocated (GiB)")
    for a in ax:
        a.tick_params(axis="x", labelsize=8)
        a.grid(axis="y", alpha=0.3)
    plt.tight_layout()
    plt.savefig(path, dpi=120)
    plt.close()


def main():
    os.makedirs("results/plots", exist_ok=True)
    out = ["# Results (auto-generated by make_report.py)\n"]
    screen = load("results/screen/*.json")
    if screen:
        out += ["## Integrator screening (5M tokens, batch 32)\n", HDR]
        out += [row(n, r) for n, r in screen.items()]
        curves(list(screen), "results/screen", "results/plots/screen_loss.png", "Integrator screening")
        out += ["\n![screening](results/plots/screen_loss.png)\n"]

    res = {n: r for n, r in load("results/*.json").items() if n in MAIN}
    if res:
        out += ["## Main runs (50M tokens)\n", HDR]
        out += [row(LABEL[n], res[n]) for n in MAIN if n in res]
        curves([n for n in MAIN if n in res], "results", "results/plots/main_loss.png", "Main runs")
        bars(res, "results/plots/main_bars.png")
        out += ["\n![loss](results/plots/main_loss.png)\n", "![bars](results/plots/main_bars.png)\n"]
        if all(k in res for k in MAIN[:3]):
            b, s, m = res["run1_baseline"], res["run2_rev_same"], res["run3_rev_max"]
            k = "tokens_per_s_steady"
            out += ["## Headline numbers\n",
                    f"- Peak memory at equal batch: {b['peak_mem_gib']:.2f} → {s['peak_mem_gib']:.2f} GiB "
                    f"(**{100 * (1 - s['peak_mem_gib'] / b['peak_mem_gib']):.0f}% lower**)",
                    f"- Throughput at equal batch: {b[k]:,.0f} → {s[k]:,.0f} tok/s "
                    f"(**{100 * (1 - s[k] / b[k]):.0f}% slower**)",
                    f"- Reversible at max batch {m['batch']}: {m[k]:,.0f} tok/s = "
                    f"**{100 * m[k] / b[k]:.0f}% of baseline** throughput, peak {m['peak_mem_gib']:.2f} GiB",
                    f"- Val loss: baseline {b['val_loss']:.3f}, rev same-batch {s['val_loss']:.3f}, "
                    f"rev max-batch {m['val_loss']:.3f}"]
            if "run1b_baseline_max" in res:
                bm = res["run1b_baseline_max"]
                out += [f"- Baseline at its own max batch {bm['batch']}: {bm[k]:,.0f} tok/s, "
                        f"val {bm['val_loss']:.3f}, peak {bm['peak_mem_gib']:.2f} GiB"]
    mb = load("results/maxbatch_*.json")
    if mb:
        out += ["\n## Max batch probes (13.5 GiB budget)\n"]
        out += [f"- {r['trunk']} (stream {r['stream']}): **{r['max_batch']}**" for r in mb.values()]
    open("REPORT_results.md", "w", encoding="utf-8").write("\n".join(out) + "\n")
    print("wrote REPORT_results.md")


if __name__ == "__main__":
    main()

## 3. Tests
The reversible backward must give the **same gradients** as ordinary autograd before any result means anything.

### `tests/test_reversible.py`

In [ ]:
%%writefile tests/test_reversible.py
import os
import sys

import pytest
import torch

sys.path.insert(0, os.path.dirname(os.path.dirname(os.path.abspath(__file__))))
from model import GPT, GPTConfig  # noqa: E402
from reversible import reconstruction_error  # noqa: E402

DEV = "cuda" if torch.cuda.is_available() else "cpu"


def tiny(trunk, n_layer=4, h=0.5):
    return GPTConfig(vocab_size=97, seq_len=16, n_layer=n_layer, n_head=2, d_model=32, trunk=trunk, h=h)


def grads(trunk, stored_sd, idx, tgt, h):
    torch.manual_seed(0)
    m = GPT(tiny(trunk, h=h)).to(DEV).double()
    m.load_state_dict(stored_sd)
    loss = m(idx, tgt)
    loss.backward()
    return loss.item(), {n: p.grad.clone() for n, p in m.named_parameters()}


@pytest.mark.parametrize("base,h", [("midpoint", 0.5), ("midpoint", 0.25), ("reveuler", 1.0)])
def test_grad_equivalence(base, h):
    torch.manual_seed(0)
    ref = GPT(tiny(base, h=h)).to(DEV).double()
    sd = ref.state_dict()
    idx = torch.randint(0, 97, (3, 16), device=DEV)
    tgt = torch.randint(0, 97, (3, 16), device=DEV)
    l1, g1 = grads(base, sd, idx, tgt, h)
    l2, g2 = grads(base + "_rev", sd, idx, tgt, h)
    assert abs(l1 - l2) < 1e-10
    for n in g1:
        assert torch.allclose(g1[n], g2[n], rtol=1e-6, atol=1e-9), n


@pytest.mark.parametrize("trunk", ["midpoint_rev", "reveuler_rev"])
def test_reconstruction(trunk):
    torch.manual_seed(0)
    m = GPT(tiny(trunk, n_layer=8)).to(DEV)
    idx = torch.randint(0, 97, (2, 16), device=DEV)
    assert reconstruction_error(m, idx) < 1e-5


@pytest.mark.skipif(DEV != "cuda", reason="needs CUDA memory stats")
def test_memory_flat_in_depth():
    def peak(trunk, L):
        torch.manual_seed(0)
        cfg = GPTConfig(vocab_size=256, seq_len=256, n_layer=L, n_head=4, d_model=256, trunk=trunk)
        m = GPT(cfg).to(DEV)
        idx = torch.randint(0, 256, (16, 256), device=DEV)
        torch.cuda.synchronize()
        torch.cuda.reset_peak_memory_stats()
        base = torch.cuda.memory_allocated()
        m(idx, idx).backward()
        torch.cuda.synchronize()
        return torch.cuda.max_memory_allocated() - base

    def act(trunk, L):  # activation part: subtract param+grad bytes
        n = sum(p.numel() for p in GPT(GPTConfig(vocab_size=256, seq_len=256, n_layer=L, n_head=4,
                                                 d_model=256, trunk=trunk)).parameters())
        return peak(trunk, L) - 4 * n

    stored = act("midpoint", 16) - act("midpoint", 4)
    rev = act("midpoint_rev", 16) - act("midpoint_rev", 4)
    assert rev < 0.2 * stored, (rev, stored)

In [ ]:
!python -m pytest tests -v

## 4. Integrator screening (5M tokens, batch 32)
Pick the integrator (and step size h) that trains stably and reaches the best loss. The stored-activation `midpoint` run is a
control: it separates architecture effects from backward effects.

In [ ]:
S = f"--tokens 5e6 --batch {B0} --dtype {DT} --out results/screen"
if RUN_SCREENING:
    !python train.py --name residual             --trunk residual $S
    !python train.py --name midpoint_h0.5        --trunk midpoint_rev --h 0.5  --stream fp64 $S
    !python train.py --name midpoint_h0.25       --trunk midpoint_rev --h 0.25 --stream fp64 $S
    !python train.py --name midpoint_h0.5_stored --trunk midpoint     --h 0.5 $S
    !python train.py --name reveuler_h1.0        --trunk reveuler_rev --h 1.0  --stream fp64 $S
    !python train.py --name reveuler_h0.5        --trunk reveuler_rev --h 0.5  --stream fp64 $S
    !python train.py --name reveuler_h0.5_fp32stream --trunk reveuler_rev --h 0.5 --stream fp32 $S

In [ ]:
# Choose the winner: best val loss among exact (fp64-stream) reversible variants.
# Default = the local screening result (reversible Euler, h = 0.5).
REV, H = "reveuler_rev", 0.5
cands = [json.load(open(p)) for p in glob.glob("results/screen/*.json")]
cands = [c for c in cands if c["trunk"].endswith("_rev") and c.get("stream") == "fp64" and not c["diverged"]]
for c in sorted(cands, key=lambda c: c["val_loss"]):
    print(f"{c['name']:22s} val {c['val_loss']:.3f}  tok/s {c['tokens_per_s_steady']:,.0f}  mem {c['peak_mem_gib']:.2f} GiB")
if cands:
    best = min(cands, key=lambda c: c["val_loss"])
    REV, H = best["trunk"], best["h"]
print("using:", REV, "h =", H)

## 5. The assignment runs (50M tokens each)

In [ ]:
# Run 1: baseline at fixed batch
!python train.py --name run1_baseline --trunk residual --batch $B0 --tokens $TOKENS --dtype $DT

In [ ]:
# Run 2: reversible at the same batch (fp64 residual stream -> exact reconstruction, exact gradients)
!python train.py --name run2_rev_same --trunk $REV --h $H --stream fp64 --batch $B0 --tokens $TOKENS --dtype $DT

In [ ]:
# Run 3: reversible at max batch
!python find_max_batch.py --trunk $REV --h $H --stream fp64 --start 64 --headroom $HEADROOM --dtype $DT
BMAX = json.load(open(f"results/maxbatch_{REV}_fp64.json"))["max_batch"]; print("max batch =", BMAX)
!python train.py --name run3_rev_max --trunk $REV --h $H --stream fp64 --batch $BMAX --scale_lr --tokens $TOKENS --dtype $DT

In [ ]:
# Optional extras: Run 1b = baseline at ITS max batch (fair payoff check); Run 3b = fp32 stream (faster, approximate)
if RUN_EXTRAS:
    !python find_max_batch.py --trunk residual --headroom $HEADROOM --dtype $DT
    B1 = json.load(open("results/maxbatch_residual_fp32.json"))["max_batch"]
    !python train.py --name run1b_baseline_max --trunk residual --batch $B1 --scale_lr --tokens $TOKENS --dtype $DT
    !python find_max_batch.py --trunk $REV --h $H --stream fp32 --start 64 --headroom $HEADROOM --dtype $DT
    B3 = json.load(open(f"results/maxbatch_{REV}_fp32.json"))["max_batch"]
    !python train.py --name run3b_rev_max_fp32stream --trunk $REV --h $H --stream fp32 --batch $B3 --scale_lr --tokens $TOKENS --dtype $DT

## 6. Report

In [ ]:
!python make_report.py
from IPython.display import Markdown, Image, display
display(Markdown(open("REPORT_results.md", encoding="utf-8").read().split("![")[0]))
for p in ["results/plots/screen_loss.png", "results/plots/main_loss.png", "results/plots/main_bars.png"]:
    if os.path.exists(p):
        display(Image(p))

## 7. Results obtained (RTX 5070 Ti 16 GB, bf16, 50M tokens per run)

| run | integrator | batch | steps | final train loss | **val loss** | **tokens/s** | **peak mem** |
|---|---|---|---|---|---|---|---|
| **Run 1** baseline (standard residual) | — | 32 | 3,052 | 1.761 | **1.849** | **243,278** | **3.22 GiB** |
| **Run 2** reversible, same batch | rev. Euler, h=0.5 | 32 | 3,052 | 1.804 | **1.890** | **156,965** | **1.16 GiB** |
| **Run 3** reversible, max batch | rev. Euler, h=0.5 | 432 | 227 | 4.760 | **3.970** | **158,349** | **10.51 GiB** |
| Run 1b (extra) baseline, max batch | — | 152 | 643 | 3.768 | 3.442 | 247,824 | 11.35 GiB |
| Run 3b (extra) reversible, max batch, **fp32 stream**, Colab T4 fp16 † | rev. Euler, h=0.5 | 704 | 139 | 5.235 | 4.497 | 43,834 † | 10.18 GiB |

† Run 3b ran on a Tesla T4 with fp16 compute; its tokens/s is not comparable with the other rows, and its gradients are approximate (reconstruction error 2.24, §6).

**Which integrator worked:** **reversible (symplectic) Euler, h = 0.5** trained stably and matched the baseline in screening
(val 3.601 vs 3.624). Midpoint/leapfrog was stable but learned more slowly (val 4.13–4.38). Its stored-activation control was
equally slow (4.21), so the gap is an architecture effect, not a backward bug.

**Findings**
* **Memory −64% at equal batch** (3.22 → 1.16 GiB). What remains is weights/grads/Adam, embeddings, one loss chunk and one layer's activations.
* **Throughput −35% at equal batch.** The backward does one extra forward per layer (≈ +33% compute in theory); about 20 points of the 35% come from
  the fp64 residual stream. An fp32 stream is faster (−20%), but its reconstruction error corrupts gradients (5.9% error for reversible Euler with h=1.0).
* **Loss ≈ unchanged** at equal batch (val 1.890 vs 1.849).
* **Max batch 2.8× larger** (432 vs 152 under the same memory budget), **but throughput was not recovered** (158K tok/s = 65% of baseline).
  The 21M model already saturates the GPU at batch 32, so this is the *compute-bound* case where reversibility is a tax (§9.1).
* Under a fixed 50M-token budget a huge batch means few optimizer steps (227 vs 3,052), so Run 3's loss is much worse. That comes from the
  step count, not from reversibility; the baseline at its max batch shows the same effect.

> *Reversibility cut peak memory ~64%, cost ~35% throughput at equal batch, and the 2.8× larger batch in Run 3 did **not** recover
> throughput (65% of baseline). The model is compute-bound, so the trade only pays off when memory is the binding constraint
> (long context, deep models, small GPUs).*

Full write-up: `REPORT.md`.

## 8. Run 3b on Google Colab (reversible, max batch, fp32 residual stream)

Run on **Tesla T4** with **fp16** autocast, using the same tokenizer and token files as the local runs (`colab_bundle.zip`), so loss is comparable across machines. Tokens/s is comparable only between rows on the same GPU.

| run | GPU | dtype | stream | batch | steps | final train loss | val loss | tokens/s (steady) | peak mem (GiB) | recon err |
|---|---|---|---|---|---|---|---|---|---|---|
| run3b_rev_max_fp32stream | Tesla T4 | fp16 | fp32 | 704 | 139 | 5.235 | 4.497 | 43,834 | 10.18 | 2.2e+00 |

Max-batch probes on this GPU: reveuler_rev stream fp32 → **704** (budget 12.9 GiB)

**Findings (auto-generated from the numbers above):**

- fp32-stream reconstruction error at the end of training: **2.24e+00** → reconstruction is **unreliable** (gradients only approximate).
- Local Run 3 (fp64 stream, NVIDIA GeForce RTX 5070 Ti, batch 432): val loss 3.970; Colab Run 3b val loss 4.497 at batch 704 (139 steps). Both are limited mainly by the small number of optimizer steps under the fixed 50M-token budget.

**Interpretation (Run 3b).**
- **It trained without diverging**, but the fp32-stream reconstruction error at the end was **2.24**, measured relative to the
  (small) input embedding x₀. The reversed states no longer match the forward ones well, so **the gradients were only
  approximate**. The discarded near-full-VRAM local attempt at batch 744 showed the same, worse (45). With the fp64 stream
  (Runs 2 and 3) the error is exactly 0.
- **Val loss 4.497 vs 3.970 for Run 3.** Two causes add up and can't be separated cleanly here: fewer optimizer steps
  (139 vs 227 under the same 50M tokens) and approximate gradients. The local runs show loss rising steeply as steps
  shrink (643 steps → 3.442, 227 → 3.970), so part of the gap is the step count.
- **What the fp32 stream buys:** a bigger max batch (704 on the T4 with a 12.9 GiB budget, against 432 for fp64 locally
  with 11.7 GiB) and ~25% more speed at equal batch locally (195K vs 157K tok/s). **What it costs:** exact gradients.
  **Recommendation: use the fp64 stream**; the fp32 stream is a speed/memory shortcut that gives up correctness.
- Precision note: Colab used **fp16** block compute (a T4 has no native bf16) against bf16 locally, with loss scaling.
  That shifts loss slightly but not the conclusions. The T4's 43.8K tok/s isn't comparable with the RTX 5070 Ti runs.
  (The final progress line printed 77,788 tok/s because of a logging bug in the last, shorter window, now fixed in
  `train.py`; the JSON value of 43,834 is correct.)